In [5]:
import re
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Inisialisasi Sastrawi (dibuat sekali di luar fungsi agar efisien saat fungsi preprocess_text() dipanggil berulang kali)
stopword_factory = StopWordRemoverFactory()
stopword_remover = stopword_factory.create_stop_word_remover()

stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

# Fungsi preprocess_text(text)
# Urutan: Case folding -> Cleaning -> Tokenisasi -> Stopwords removal -> Stemming
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    text_no_stopwords = stopword_remover.remove(' '.join(tokens))
    tokens_no_stopwords = text_no_stopwords.split()
    stemmed_tokens = [stemmer.stem(t) for t in tokens_no_stopwords]

    return stemmed_tokens

# Dataset: 5 dokumen berita/pengumuman Fakultas Teknik UNM (tema teknologi, komputer, dan pendidikan)
dokumen = [
    # Dokumen 1
    "Fakultas Teknik UNM mengumumkan bahwa pendaftaran mahasiswa baru Program Studi Teknik Komputer telah dibuka mulai tanggal 1 Oktober 2026. Calon mahasiswa diharapkan segera melengkapi berkas administrasi!",
    # Dokumen 2
    "Sistem temu kembali informasi merupakan salah satu topik penting dalam bidang ilmu komputer, yang mempelajari bagaimana cara menemukan dokumen-dokumen yang relevan terhadap sebuah query pencarian.",
    # Dokumen 3
    "Laboratorium Jaringan Komputer Fakultas Teknik akan mengadakan pelatihan keamanan siber (cyber security) bagi mahasiswa semester 5 pada tanggal 12-14 November 2026, bertempat di Gedung C lantai 3.",
    # Dokumen 4
    "Perkembangan teknologi kecerdasan buatan (Artificial Intelligence) saat ini berkembang sangat pesat, terutama pada bidang pengolahan bahasa alami dan sistem rekomendasi yang digunakan oleh banyak perusahaan.",
    # Dokumen 5
    "Pengumuman: Ujian Tengah Semester mata kuliah Sistem Temu Kembali Informasi akan dilaksanakan secara daring melalui platform kampus mulai pukul 08.00 WITA, mahasiswa wajib hadir tepat waktu!",
]

nama_dokumen = [f"Dokumen {i+1}" for i in range(len(dokumen))]

# Terapkan preprocess_text() pada ke-5 dokumen
hasil_preprocessing = [preprocess_text(doc) for doc in dokumen]

# Perbandingan sebelum & sesudah preprocessing (ditampilkan lengkap untuk 2 dokumen pertama, sesuai ketentuan soal)
print("=" * 80)
print("PERBANDINGAN SEBELUM DAN SESUDAH PREPROCESSING")
print("=" * 80)

for i in range(2):
    print(f"\n--- {nama_dokumen[i]} ---")
    print("Sebelum preprocessing :")
    print(dokumen[i])
    print("\nSesudah preprocessing (token) :")
    print(hasil_preprocessing[i])

print("\n" + "=" * 80)
print("RINGKASAN SELURUH DOKUMEN (Dokumen 1-5)")
print("=" * 80)
for i in range(len(dokumen)):
    print(f"\n--- {nama_dokumen[i]} ---")
    print("Sebelum :", dokumen[i])
    print("Sesudah :", hasil_preprocessing[i])

# Hitung jumlah token sebelum & sesudah preprocessing, serta persentase pengurangan token

rekap = []
for i in range(len(dokumen)):
    jumlah_token_sebelum = len(dokumen[i].split())          # tokenisasi naif (spasi)
    jumlah_token_sesudah = len(hasil_preprocessing[i])       # hasil akhir pipeline
    persentase_pengurangan = (
        (jumlah_token_sebelum - jumlah_token_sesudah) / jumlah_token_sebelum * 100
    )
    rekap.append({
        "Dokumen": nama_dokumen[i],
        "Jumlah Token Sebelum": jumlah_token_sebelum,
        "Jumlah Token Sesudah": jumlah_token_sesudah,
        "Persentase Pengurangan (%)": round(persentase_pengurangan, 2),
    })

df_rekap = pd.DataFrame(rekap)

print("\n" + "=" * 80)
print("TABEL JUMLAH TOKEN SEBELUM & SESUDAH PREPROCESSING")
print("=" * 80)
print(df_rekap.to_string(index=False))

rata_rata_pengurangan = df_rekap["Persentase Pengurangan (%)"].mean()
print(f"\nRata-rata persentase pengurangan token: {rata_rata_pengurangan:.2f}%")

# Analisis singkat
analisis = f"""
ANALISIS SINGKAT
-----------------
Preprocessing (case folding, cleaning, tokenisasi, stopwords removal, dan stemming) terbukti mengurangi jumlah token pada kelima
dokumen rata-rata sekitar {rata_rata_pengurangan:.2f}%. Pengurangan ini terutama berasal dari penghapusan tanda baca, angka, dan kata-kata umum (stopwords) seperti "yang",
"dan", "di", "untuk" yang sering muncul namun tidak membawa informasi diskriminatif, serta penyatuan kata-kata berimbuhan menjadi bentuk dasarnya
lewat proses stemming (misalnya kata berawalan "me-" atau berakhiran "-kan" diubah menjadi kata dasarnya). Bagi sistem temu kembali informasi, hal ini sangat bermanfaat
karena vocabulary yang terbentuk menjadi lebih kecil dan lebih fokus pada term-term yang benar-benar merepresentasikan topik dokumen, sehingga proses
pembobotan (misalnya TF-IDF) pada tahap berikutnya dapat menghasilkan ranking dokumen yang lebih akurat dan efisien secara komputasi.
"""
print(analisis)

PERBANDINGAN SEBELUM DAN SESUDAH PREPROCESSING

--- Dokumen 1 ---
Sebelum preprocessing :
Fakultas Teknik UNM mengumumkan bahwa pendaftaran mahasiswa baru Program Studi Teknik Komputer telah dibuka mulai tanggal 1 Oktober 2026. Calon mahasiswa diharapkan segera melengkapi berkas administrasi!

Sesudah preprocessing (token) :
['fakultas', 'teknik', 'unm', 'umum', 'daftar', 'mahasiswa', 'baru', 'program', 'studi', 'teknik', 'komputer', 'buka', 'mulai', 'tanggal', 'oktober', 'calon', 'mahasiswa', 'harap', 'segera', 'lengkap', 'berkas', 'administrasi']

--- Dokumen 2 ---
Sebelum preprocessing :
Sistem temu kembali informasi merupakan salah satu topik penting dalam bidang ilmu komputer, yang mempelajari bagaimana cara menemukan dokumen-dokumen yang relevan terhadap sebuah query pencarian.

Sesudah preprocessing (token) :
['sistem', 'temu', 'informasi', 'rupa', 'salah', 'satu', 'topik', 'penting', 'bidang', 'ilmu', 'komputer', 'ajar', 'bagaimana', 'cara', 'temu', 'dokumen', 'dokumen', 'relev